In [1]:
#python version
import sys
print(sys.version)

3.8.18 (default, Sep 11 2023, 13:47:48) [MSC v.1916 64 bit (AMD64)]


In [ ]:
# ===============================================
# Preprocessing part1 # MIMIC-4 data clean
# ===============================================
#===============================
#-------------------------------
'''Input Data : one row is one admission
[
    'subject_id',   # Unique patient identifier (Static across all longitudinal admissions)
    'hadm_id',      # Unique encounter/admission identifier for the index hospitalization
    'icd3_list',    # List of 3-digit ICD diagnosis codes recorded during the index admission(ex.["E11","N18","I25"...])
    'admittime',    # Timestamp of hospital admission for the index episode
    'deathtime',    # Timestamp of patient death (Null if the patient survived/censored)
    'los',          # Length of stay for the current hospitalization (Measured in days)
    'age',          # Patient age at the time of the index admission (Baseline chronological age)
    'gender'        # Patient biological sex at birth (Demographic baseline factor)
]
'''
#-------------------------------
#===============================

In [ ]:
# ===============================================
# Preprocessing part2 # label and 3-Y Path
# ===============================================

In [ ]:
import pandas as pd
import ast

df_sample = pd.read_csv("./sample_data.csv")

#把ICD字串轉回真正的 Python list
df_sample["icd3_list"] = df_sample["icd3_list"].apply(ast.literal_eval)

#確保 los , age 欄位是整數 (int)
df_sample["age"] = pd.to_numeric(df_sample["age"], errors="coerce").astype(int)
df_sample["los"] = pd.to_numeric(df_sample["los"], errors="coerce").astype(int)

#確保時間欄位是 datetime
df_sample["admittime"] = pd.to_datetime(df_sample["admittime"])

In [ ]:
# ===============================================
# Schema : 'icd_filterZ',  # Refined ICD-3 list excluding non-specific codes (Z-codes filtered for uncertainty control)
# ===============================================
#移除Zcode: remove_Z=True
def filter_icds(icds, remove_Z=False):
    if remove_Z:
        return [c for c in icds if not c.startswith("Z")]
    return icds

df_sample["icd_filterZ"] = df_sample["icd3_list"].apply(
    lambda x: filter_icds(x, remove_Z=True)
)

# 只保留原始清單icd_filterZ不為空的就診 #icd3_list
df_sample = df_sample[df_sample["icd_filterZ"].map(len) > 0].reset_index(drop=True)
df_sample

In [ ]:
# ==========================
# label 設定
# ==========================
target_icd = "I50"  # 目標疾病 ICD，可自行更換

# 結構性 ICD（不參與模型，避免被結果碼主導）
STRUCTURAL_ICDS = {target_icd,"I11","I13","I42","I43","I51","J81"}  # 可以擴充其他 ICD

# ==========================
# 複製資料，避免 SettingWithCopyWarning
# ==========================
df_pre = df_sample.copy()

# ==========================
# 標記目標 ICD
# ==========================
df_pre['label'] = df_pre['icd3_list'].apply(lambda icds: 1 if target_icd in icds else 0)

# ==========================
# 清理結構性 ICD (不參與模型，避免被結果碼主導)
# ==========================
def clean_icds(icds, remove_icds):
    return [c for c in icds if c not in remove_icds]

df_pre["icd_clean"] = df_pre["icd_filterZ"].apply(lambda icds: clean_icds(icds, STRUCTURAL_ICDS))
df_sample["icd_clean"] = df_sample["icd_filterZ"].apply(lambda icds: clean_icds(icds, STRUCTURAL_ICDS))

# ==========================
# 按時間排序
# ==========================
df_pre = df_pre.sort_values("admittime").reset_index(drop=True)

#刪掉重複的hadm_id
df_pre = (
    df_pre
    .sort_values("admittime")
    .drop_duplicates(subset="hadm_id", keep="last")
)


In [ ]:
#==============================
#計算時間窗的PATH ICD
#==============================
#建立 pregnancy-level ICD path 「每次住院一個 list」
from datetime import timedelta

def build_pregnancy_path_visits_and_time(row, adm_df, window_days=365*3):

    pid = row["subject_id"]
    index_time = row["admittime"]
    index_hadm = row["hadm_id"]

    hist = adm_df[
        (adm_df["subject_id"] == pid) &
        (adm_df["admittime"] < index_time) &
        (adm_df["admittime"] >= index_time - timedelta(days=window_days)) &
        (adm_df["hadm_id"] != index_hadm)
    ].sort_values("admittime")

    visit_paths = []
    visit_paths_clean = []
    visit_times = []
    visit_days = []

    for _, r in hist.iterrows():

        visit_paths.append(r["icd_filterZ"]) #icd_clean,icd_filterZ
        visit_paths_clean.append(r["icd_clean"])
        visit_times.append(r["admittime"])

        delta_days = (r["admittime"] - index_time).days
        visit_days.append(delta_days)

    return visit_paths, visit_times, visit_days, visit_paths_clean

#套用到 df_pre
result = df_pre.apply(
    lambda row: build_pregnancy_path_visits_and_time(row, df_sample),
    axis=1
)

df_pre["path"] = result.apply(lambda x: x[0])
df_pre["path_time"] = result.apply(lambda x: x[1])
df_pre["path_day"] = result.apply(lambda x: x[2])
df_pre["path_clean"] = result.apply(lambda x: x[3])
df_pre["path_len"] = df_pre["path"].apply(len)


In [ ]:
#刪掉沒有path的資料
df_pre = df_pre[df_pre["path_len"]>0].copy()

print("新 df_pro 筆數:", len(df_pre))
print("Data label dist:\n", df_pre['label'].value_counts(normalize=True))
print(df_pre["path_len"].describe())

In [ ]:
# =========================================================
# 1. Basic functions
# =========================================================

In [ ]:
# =========================================================
# mean/max pooling
# =========================================================
import numpy as np

def safe_mean_pooling(icds, icd_embeddings, embed_dim=64):
    vecs = [icd_embeddings[c] for c in icds if c in icd_embeddings]
    if len(vecs) == 0:
        return np.zeros(embed_dim)
    return np.mean(vecs, axis=0)

def safe_max_pooling(icds, icd_embeddings, embed_dim=64):
    vecs = [icd_embeddings[c] for c in icds if c in icd_embeddings]
    if len(vecs) == 0:
        return np.zeros(embed_dim)
    return np.max(vecs, axis=0)

In [ ]:
# =========================================================
# IG-weight
# =========================================================
# Entropy
# =========================================================
def entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    return -p*np.log2(p) - (1-p)*np.log2(1-p)

# =========================================================
#transform ig:log + normalization
#normalize 保證最大值是 1，最小值是 0(排序不變，重要 ICD 得到高權重)
# =========================================================
def transform_ig(ig_dict):
    eps=1e-8 #防止 log(0)
    
    ig_vals = np.array(list(ig_dict.values()))
    log_ig = np.log(ig_vals + eps)

    # normalize to [0, 1]
    log_ig = (log_ig - log_ig.min()) / (log_ig.max() - log_ig.min())

    return dict(zip(ig_dict.keys(), log_ig))

# =========================================================
# IG-weighted pooling
# =========================================================
def ig_weighted_pooling(icd_list,embedding_dict,ig_dict,default_ig):
    vectors = []
    weights = []

    for icd in icd_list:
        if icd not in embedding_dict:
            continue  # graph 沒出現的 ICD，直接跳過(baseline)=新 ICD 泛化能力 = 0(目前)

        #「有出現在圖裡（有 Embedding），但因為特殊原因在訓練集算出來的資訊增益等於 0 或被過濾掉，導致不存在於 ig_dict 裡」的已知 ICD，它才會拿到 default_ig
        w = ig_dict.get(icd, default_ig)
        vectors.append(embedding_dict[icd])
        weights.append(w)

    if len(vectors) == 0:
        return np.zeros(len(next(iter(embedding_dict.values()))))

    vectors = np.vstack(vectors)
    weights = np.array(weights)

    # --- 修正安全機制 ---
    # 檢查權重絕對值的總和是否為 0（或極接近 0）
    if np.sum(np.abs(weights)) == 0:
        # 退回一般的平均（不加權）
        pooled = np.mean(vectors, axis=0)
        
        return pooled
    # --------------------
    
    # normalized weighted mean 加權平均
    pooled = np.average(vectors, axis=0, weights=weights)
    return pooled

# =========================================================
#IG-GATED MAX POOLING
# =========================================================
def ig_gated_max_pooling(icd_list, embedding_dict, ig_dict, default_ig):
    eps=1e-6
    
    vecs = []
    gates = []

    for icd in icd_list:
        if icd not in embedding_dict:
            continue

        ig = ig_dict.get(icd, default_ig)

        # IG gate（關鍵）
        gate = np.log1p(ig) + 1.0   # >= 1

        vecs.append(embedding_dict[icd])
        gates.append(gate)

    if len(vecs) == 0:
        return np.zeros(len(next(iter(embedding_dict.values()))))

    E = np.vstack(vecs)        # (N, D)
    G = np.array(gates)[:, None]  # (N, 1)

    gated = E * G
    pooled = np.max(gated, axis=0)

    return pooled

In [ ]:
# =========================================================
# visit-pooling : 將每次住院的pooling再聚合起來 
# =========================================================

In [ ]:
# =========================================================
# 1.visit-pooling : mean /max only
# ICD mean完再 mean visits，ICD max完再 max visits
# =========================================================
#path mean
def path_meanmean_to_embedding(path, embedding_dict, ig_dict, default_ig):
    vecs = []

    for visit_icds in path:
        # 1. mean pooling
        v_mean = safe_mean_pooling(
            visit_icds,
            embedding_dict
        )
        vecs.append(v_mean)
    
    if len(vecs) == 0:
        return np.zeros(len(next(iter(embedding_dict.values()))))

    return np.mean(np.vstack(vecs), axis=0)

#path max-max
def path_maxmax_to_embedding(path, embedding_dict, ig_dict, default_ig):
    vecs = []

    for visit_icds in path:
        # 1. max pooling
        v_max = safe_max_pooling(
            visit_icds,
            embedding_dict
        )
        vecs.append(v_max)
    
    if len(vecs) == 0:
        return np.zeros(len(next(iter(embedding_dict.values()))))

    return np.max(np.vstack(vecs), axis=0)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
#Geometry
def calc_geometry_features(mean_mat):
    deviations = mean_mat - mean_mat.mean(axis=0)
    var_dist = np.mean(np.sum(deviations**2, axis=1))

    cos_mat = cosine_similarity(mean_mat)
    n = len(mean_mat)
    mean_cos = (cos_mat.sum() - n) / (n*(n-1)) if n > 1 else 1.0

    return np.array([var_dist, mean_cos])


def cosine_mean_and_var(mean_mat):
    """
    mean_mat: (n_visits, dim)
    return:
        mean_cos: 平均 pairwise cosine similarity
        var_cos : pairwise cosine similarity 的變異性（std）
    """
    n = len(mean_mat)
    if n <= 1:
        return 1.0, 0.0

    cos_mat = cosine_similarity(mean_mat)
    # 只取上三角（不含對角線)
    vals = cos_mat[np.triu_indices(n, k=1)]

    return np.mean(vals), np.std(vals)


def cosine_sim(a, b, eps=1e-8):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)


def cosine_mean_var_to_centroid(mat, centroid):
    """
    mat: (T, d) matrix, e.g. ig_mat
    centroid: (d,) vector, e.g. mean_vec_pos
    """
    sims = [
        cosine_sim(v, centroid)
        for v in mat
    ]
    sims = np.array(sims)
    return sims.mean(), sims.var()

In [ ]:
def path_to_embedding_fun(
    path, path_day, embedding_dict, 
    ig_dict=None, default_ig=1.0, mean_vec_pos=None,
    use_ig=True,         # True: 使用 IGwGated, False: 使用傳統 Max
    use_time_weight=True, # True: 計算時間權重, False: 均值平均 (noTime)
    include_step=True,   # True: 包含時序軌跡特徵 (Step)
    include_geometry=True # True: 包含幾何與空間特徵 (Geometry)
):
    """
    【特徵工程函數】
    """
    d = len(next(iter(embedding_dict.values())))
    
    # 1. 基礎 Visit 級 Pooling 抽取
    mean_vecs = []
    target_vecs = [] # 根據參數決定是存 max_vecs 還是 ig_vecs
    
    for visit_icds in path:
        v_mean = safe_mean_pooling(visit_icds, embedding_dict)
        mean_vecs.append(v_mean)
        
        # 決定使用資訊增益(IG)還是傳統最大值(Max)
        if use_ig and (ig_dict is not None):
            v_target = ig_gated_max_pooling(visit_icds, embedding_dict, ig_dict, default_ig)
        else:
            v_target = safe_max_pooling(visit_icds, embedding_dict)
        target_vecs.append(v_target)

    # 空白病史防呆
    if len(mean_vecs) == 0:
        # 動態計算回傳維度大小
        total_dim = 4 * d
        if include_step: total_dim += 2 * d
        if include_geometry: total_dim += 5
        return np.zeros(total_dim)

    # 矩陣化
    mean_mat = np.vstack(mean_vecs)
    target_mat = np.vstack(target_vecs)
    diff_mat = target_mat - mean_mat

    # 2. 時間權重計算
    if use_time_weight and path_day is not None:
        WINDOW = 365 * 3
        days = np.abs(np.array(path_day))
        raw_weights = np.clip(1 - days / WINDOW, 0.01, None)
        norm_weights = raw_weights / raw_weights.sum()
    else:
        norm_weights = None # np.average 傳入 None 即為傳統不加權平均 (noTime)

    # 3. 基礎四大核心特徵抽取
    path_mean = np.average(mean_mat, axis=0, weights=norm_weights)
    path_target_max = target_mat.max(axis=0)
    path_diffsum = diff_mat.sum(axis=0)
    path_diffmean = np.average(diff_mat, axis=0, weights=norm_weights)
    
    feature_chunks = [path_mean, path_target_max, path_diffsum, path_diffmean]

    # 4. 模組化加入：時序軌跡特徵 (Step / Trajectory)
    if include_step:
        if len(mean_mat) >= 2:
            visit_delta = mean_mat[1:] - mean_mat[:-1]
            trajectory_change_sum = np.abs(visit_delta).sum(axis=0)
            trajectory_change_mean = np.average(np.abs(visit_delta), axis=0)
        else:
            trajectory_change_sum = np.zeros(d)
            trajectory_change_mean = np.zeros(d)
        feature_chunks.extend([trajectory_change_sum, trajectory_change_mean])

    # 5. 模組化加入：空間幾何特徵 (Geometry)
    if include_geometry:
        # 計算變異量
        deviations = (target_mat if use_ig else mean_mat) - path_mean
        var_dist = np.mean(np.sum(deviations ** 2, axis=1))
        #log_var_dist = np.log1p(var_dist)
        
        # 內部餘弦相似度
        mean_cos, var_cos = cosine_mean_and_var(target_mat if not use_ig else target_mat)
        
        # 與正樣本質心的餘弦相似度
        mean_cos_pos, var_cos_pos = cosine_mean_var_to_centroid(
            mean_mat if not use_ig else target_mat, 
            mean_vec_pos
        )
        geometry_features = [var_dist, mean_cos, var_cos, mean_cos_pos, var_cos_pos]
        feature_chunks.append(geometry_features)

    # 6. 拼接
    return np.concatenate(feature_chunks)

In [ ]:
# =========================================================
# 0. Imports & basic setup
# =========================================================
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
import math
import networkx as nx
from node2vec import Node2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from prefixspan import PrefixSpan

from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.base import clone

from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score
)

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set(style="whitegrid")


MIN_COOC = 3
USE_PPMI = True
EMBED_DIM = 64
N_SPLITS = 5 #10

df = df_pre.copy()

# 同一個 subject 不會跨 fold，且每 fold 的比例差不多
from sklearn.model_selection import StratifiedGroupKFold

groups_sgkf = df["subject_id"]
y = df["label"]

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

In [ ]:
#「訓練模型 -> 預測機率 -> 尋找最佳 Threshold -> 記錄 CV 結果」
def evaluate_classifier(X_tr, y_tr, X_te, y_te, model, model_name, col_name, fold, test_pos_idx_ml):
    """
    分類器評估與記錄
    """
    # 針對 XGB 這種需要動態算 scale_pos_weight 的模型做處理
    if model_name == "XGB" and hasattr(model, "scale_pos_weight"):
        neg, pos = np.bincount(y_tr)
        model.set_params(scale_pos_weight=neg / pos)
        
    # 訓練與預測
    model.fit(X_tr, y_tr)
    y_prob = model.predict_proba(X_te)[:, 1]

    # 1. 存入 OOF 與 Fold Predictions
    oof_prob[model_name][col_name][test_pos_idx_ml] = y_prob
    pr_fold_predictions[model_name][col_name].append({
        "fold": fold, "y_true": y_te.copy(), "y_prob": y_prob.copy()
    })

    # 2. 向量化尋找最佳 Threshold 
    thresholds = np.linspace(0.05, 0.95, 181)
    # [n_samples, n_thresholds]
    y_pred_matrix = y_prob_xgb = (y_prob_xgb := y_prob_xgb if 'y_prob_xgb' in locals() else y_prob)[:, None] >= thresholds
    f1_scores = [f1_score(y_te, y_pred_matrix[:, i], zero_division=0) for i in range(len(thresholds))]
    best_idx = np.argmax(f1_scores)
    best_f1, best_th = f1_scores[best_idx], thresholds[best_idx]
    y_pred_best = y_pred_matrix[:, best_idx]

    # 3. 紀錄至對應的 Summary 容器
    summary_target = cv_summary if model_name == "XGB" else cv_summary_lr
    summary_target[col_name]["best_f1"].append(best_f1)
    summary_target[col_name]["best_threshold"].append(best_th)
    summary_target[col_name]["AUC"].append(roc_auc_score(y_te, y_prob))
    summary_target[col_name]["AUPRC"].append(average_precision_score(y_te, y_prob))
    summary_target[col_name]["classification_report"].append(classification_report(y_te, y_pred_best, digits=3, zero_division=0))
    summary_target[col_name]["confusion_matrix"].append(confusion_matrix(y_te, y_pred_best))

    print(f"[{model_name} - {col_name}] Best TH: {best_th:.2f} | F1: {best_f1:.3f} | AUC: {summary_target[col_name]['AUC'][-1]:.3f} | AUPRC: {summary_target[col_name]['AUPRC'][-1]:.3f}")

In [ ]:
#重構後的 10-Fold 主迴圈
# =====================================================================
# 0. 定義消融實驗配置 (Ablation Configuration)
# =====================================================================
# 格式: 特徵欄位名稱 -> (use_ig, use_time, include_step, include_geometry)
FEAT_CONFIGS = {
    #"mean-mean":                 (False, False, False, False), # 對照組
    #"max-max":                  (False, False, False, False),
    #"mean-maxDiff":              (False, True,  False, False),
    #"mean-maxDiffStep":          (False, True,  True,  False),
    #"mean-maxDiffGeometry":      (False, True,  False, True),
    "mean-maxDiffGeometryStep":  (False, True,  True,  True),
    #"mean-IGwDiff":              (True,  True,  False, False),
    #"mean-IGwDiffStep":          (True,  True,  True,  False),
    #"mean-IGwDiffGeometry":      (True,  True,  False, True),
    "mean-IGwDiffGeometryStep":  (True,  True,  True,  True), # 核心
}

# 所有的方法
ALL_METHODS = list(FEAT_CONFIGS.keys())

# 初始化統計容器
oof_prob = {mod: {met: np.full(len(df), np.nan) for met in ALL_METHODS} for mod in ["XGB", "LR"]}
cv_summary = {met: defaultdict(list) for met in ALL_METHODS}
cv_summary_lr = {met: defaultdict(list) for met in ALL_METHODS}
pr_fold_predictions = {mod: {met: [] for met in ALL_METHODS} for mod in ["XGB", "LR"]}


fold_list = []

# =====================================================================
# 主迴圈開始 10-CV
# =====================================================================
for fold, (train_idx, test_idx) in enumerate(sgkf.split(df, y, groups_sgkf), start=1):
    print(f"\n================ Fold {fold} =================\n")
    
    df_train = df.iloc[train_idx].copy()
    df_test  = df.iloc[test_idx].copy()

    # 確保沒有病人洩漏 (Data Leakage Check)
    assert len(set(df_train["subject_id"]) & set(df_test["subject_id"])) == 0
    fold_list.append(pd.DataFrame({"row_idx": test_idx, "subject_id": df_test["subject_id"].values, "label": df_test["label"].values, "fold": fold}))

    # -----------------------------------------------------------------
    # [IG 與 Node2Vec 計算] (計算出 icd_embeddings, ig_dict, default_ig, mean_vec_pos)
    # -----------------------------------------------------------------
    # -------------------------------------------------
    # IG (train-only)
    # -------------------------------------------------
    all_icds = sorted({icd for icds in df_train["icd_clean"] for icd in icds})

    N = len(df_train)
    N_pos = (df_train["label"] == 1).sum()
    H_y = entropy(N_pos / N)

    ig_dict = {}
    for icd in all_icds:
        mask = df_train["icd_clean"].apply(lambda x: icd in x)
        N1 = mask.sum()
        if N1 == 0 or N1 == N:
            continue

        N1_pos = ((mask) & (df_train["label"] == 1)).sum()
        N0_pos = N_pos - N1_pos

        H_y_icd = (
            (N1 / N) * entropy(N1_pos / N1) +
            ((N - N1) / N) * entropy(N0_pos / (N - N1))
        )
        ig_dict[icd] = H_y - H_y_icd

    # 先將 ig_dict 進行 log 與 0~1 歸一化轉換
    ig_dict = transform_ig(ig_dict)

    # 轉換完後，再從轉換後的 dict 裡面計算第 25 百分位數作為 default_ig
    transformed_ig_vals = np.array(list(ig_dict.values()))
    default_ig = np.percentile(transformed_ig_vals, 25)
    
    # 看前 10 個 IG 最大的 ICD
    print(sorted(ig_dict.items(), key=lambda x: x[1], reverse=True)[:10])

    # -------------------------------------------------
    # Build train-only PMI / PPMI graph
    # -------------------------------------------------
    docs = df_train["icd_clean"]
    N_docs = len(docs)

    df_icd = Counter()
    df_pair = Counter()

    for icds in docs:
        icds = list(set(icds))
        for c in icds:
            df_icd[c] += 1
        for a, b in combinations(sorted(icds), 2):
            df_pair[(a, b)] += 1

    G = nx.Graph()
    for (a, b), cij in df_pair.items():
        if cij < MIN_COOC:
            continue
        pi = df_icd[a] / N_docs
        pj = df_icd[b] / N_docs
        pij = cij / N_docs
        pmi = math.log(pij / (pi * pj + 1e-12) + 1e-12)
        if USE_PPMI:
            pmi = max(pmi, 0.0)
            if pmi == 0:
                continue
        G.add_edge(a, b, weight=pmi)

    #某個ICD的所有鄰居，建成LIST
    graph_neighbors = {
        node: list(G.neighbors(node))
        for node in G.nodes()
    }
    print("Graph nodes:", G.number_of_nodes())

    # -------------------------------------------------
    # Node2Vec
    # -------------------------------------------------
    node2vec = Node2Vec(
        G,
        dimensions=EMBED_DIM,
        walk_length=10,
        num_walks=30,
        workers=1,
        seed=SEED,
        weight_key="weight",
        quiet=True
    )
    w2v_model = node2vec.fit(window=5, min_count=1)
    icd_embeddings = {n: w2v_model.wv[n] for n in G.nodes}
    
    # 篩選有效病史，拿掉path=[](為0的)
    train_mask, test_mask = df_train["path"].apply(len) > 0, df_test["path"].apply(len) > 0
    df_train_ml, df_test_ml = df_train[train_mask].copy(), df_test[test_mask].copy()

    # 再取出 path>0 的那些位置
    test_pos_idx_ml = df.index.get_indexer(df_test.index)[test_mask.values]
    y_tr, y_te = df_train_ml["label"].astype(int).values, df_test_ml["label"].astype(int).values

    #定義「正類的幾何中心」
    vecs_pos = (
        df_train_ml
        .loc[df_train_ml["label"] == 1, "path"]
        .apply(lambda r: path_meanmean_to_embedding(
            r, icd_embeddings, ig_dict, default_ig
        ))
        .values
    )
    mean_vec_pos = np.mean(np.vstack(vecs_pos), axis=0)

    # -----------------------------------------------------------------
    # 使用函數生成所有常規特徵
    # -----------------------------------------------------------------
    print("生成特徵矩陣")
    X_train_dict, X_test_dict = {}, {}
    
    for col, params in FEAT_CONFIGS.items():
        # 這裡直接呼叫消融函數：
        df_train_ml[col] = df_train_ml.apply(lambda r: path_to_embedding_fun(r["path"], r["path_day"], icd_embeddings, ig_dict, default_ig, mean_vec_pos, *params), axis=1)
        df_test_ml[col]  = df_test_ml.apply(lambda r: path_to_embedding_fun(r["path"], r["path_day"], icd_embeddings, ig_dict, default_ig, mean_vec_pos, *params), axis=1)
        
        X_train_dict[col] = np.vstack(df_train_ml[col].values)
        X_test_dict[col]  = np.vstack(df_test_ml[col].values)

    
    # 將所有特徵統一歸入一個大字典中管理
    all_data_dict = {**{col: (X_train_dict[col], X_test_dict[col]) for col in FEAT_CONFIGS.keys()}}

    # -----------------------------------------------------------------
    # 機器學習分類器迴圈
    # -----------------------------------------------------------------
    print("\n機器學習模型實驗")

    neg, pos = np.bincount(y_tr)
    scale_pos_weight = neg / pos
    
    # 初始化基底分類器範本 (不帶 Fold 狀態)
    xgb_template = XGBClassifier(n_estimators=400, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="logloss", random_state=SEED, reg_alpha=0.3, reg_lambda=2.0, n_jobs=4)
    lr_template = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000, class_weight="balanced", random_state=SEED, n_jobs=4)

    # 直接跑完所有特徵組合與所有模型
    for col_name, (X_tr_flat, X_te_flat) in all_data_dict.items():
        # 跑 XGB
        evaluate_classifier(X_tr_flat, y_tr, X_te_flat, y_te, clone(xgb_template), "XGB", col_name, fold, test_pos_idx_ml)
        # 跑 LR
        evaluate_classifier(X_tr_flat, y_tr, X_te_flat, y_te, clone(lr_template), "LR", col_name, fold, test_pos_idx_ml)

In [ ]:
# 重構後的 Summary
print("\n" + "="*50)
print("10-Fold CV")
print("="*50)

# 所有特徵工程實驗
for col in ALL_METHODS:
    print(f"\n--- {col} ---")
    for metric in ["AUC", "AUPRC", "best_f1"]:
        # XGB
        xgb_vals = cv_summary[col][metric]
        print(f"XGB: {metric:<8}: {np.mean(xgb_vals):.4f} ± {np.std(xgb_vals):.3f}")
        # LR
        lr_vals = cv_summary_lr[col][metric]
        print(f"LR:  {metric:<8}: {np.mean(lr_vals):.4f} ± {np.std(lr_vals):.3f}")